In [214]:
import pandas as pd

In [215]:
sales_df = pd.read_csv('../data/challenge_449_sales_figures.csv')

sales_df.head()

,Region,Date,Segment,Country,Currency,Product,Discount Band,Units Sold,Manufacturing Price,Sale Price,Gross Sales,Discounts,Sales,COGS,Profit
0,APAC,2022-02-01,Small Business,Japan,JPY,VTT,Low,1778.0,260.0,350.0,622300.0,24892.0,597408.0,462280.0,135128.0
1,APAC,2022-02-01,Enterprise,China,CNY,Velo,Medium,1802.0,10.0,20.0,36040.0,1802.0,34238.0,18020.0,16218.0
2,APAC,2022-02-01,Enterprise,Japan,JPY,Velo,Medium,2436.0,250.0,300.0,730800.0,43848.0,686952.0,609000.0,77952.0
3,APAC,2022-02-01,Government,China,CNY,Montana,Medium,1421.0,120.0,20.0,28420.0,1989.4,26430.6,14210.0,12220.6
4,APAC,2022-02-01,Government,China,CNY,Amarilla,High,808.0,250.0,300.0,242400.0,19392.0,223008.0,202000.0,21008.0


In [216]:
ex_df = pd.read_csv('../data/challenge_449_exchange_rates.csv')[['Currency', 'EUR']]

ex_df.head()

,Currency,EUR
0,EUR,1.0000
1,USD,0.9300
2,JPY,0.0063
3,GBP,1.1700
4,CHF,1.0200


In [217]:
# convert string to date, pull out year and filter unneeded rows

sales_df['Date'] = pd.to_datetime(sales_df['Date'])
sales_df['year'] = sales_df['Date'].dt.year

sales_df['Sales'].dropna(inplace=True)

sales_df.head()

,Region,Date,Segment,Country,Currency,Product,Discount Band,Units Sold,Manufacturing Price,Sale Price,Gross Sales,Discounts,Sales,COGS,Profit,year
0,APAC,2022-02-01,Small Business,Japan,JPY,VTT,Low,1778.0,260.0,350.0,622300.0,24892.0,597408.0,462280.0,135128.0,2022
1,APAC,2022-02-01,Enterprise,China,CNY,Velo,Medium,1802.0,10.0,20.0,36040.0,1802.0,34238.0,18020.0,16218.0,2022
2,APAC,2022-02-01,Enterprise,Japan,JPY,Velo,Medium,2436.0,250.0,300.0,730800.0,43848.0,686952.0,609000.0,77952.0,2022
3,APAC,2022-02-01,Government,China,CNY,Montana,Medium,1421.0,120.0,20.0,28420.0,1989.4,26430.6,14210.0,12220.6,2022
4,APAC,2022-02-01,Government,China,CNY,Amarilla,High,808.0,250.0,300.0,242400.0,19392.0,223008.0,202000.0,21008.0,2022


In [218]:
# calculate total sales in euros

sales_df = sales_df.groupby(['Region', 'Segment', 'year', 'Currency'])['Sales'].sum().reset_index()

df = pd.merge(sales_df, ex_df, how='inner', on='Currency')
df['Sales in Euros'] = round(df['Sales'] * df['EUR'], 2)

df.head()

,Region,Segment,year,Currency,Sales,EUR,Sales in Euros
0,APAC,Channel Partners,2022,CNY,2122081.820,0.1300,275870.64
1,APAC,Channel Partners,2022,JPY,3309442.170,0.0063,20849.49
2,APAC,Channel Partners,2023,CNY,2527419.340,0.1300,328564.51
3,APAC,Channel Partners,2023,JPY,4154114.345,0.0063,26170.92
4,APAC,Enterprise,2022,CNY,2031647.690,0.1300,264114.20


In [219]:
# Output 1. pivot and fix names

df = df.pivot_table(columns=['year'], values=['Sales in Euros'], index=['Region', 'Segment'], aggfunc='sum').sort_values(['Segment', 'Region']).reset_index()
df.columns = [f'{a}{b}' for a, b in df.columns]

df.head()

,Region,Segment,Sales in Euros2022,Sales in Euros2023
0,APAC,Channel Partners,296720.13,354735.43
1,Europe,Channel Partners,17478161.64,16522592.10
2,USA,Channel Partners,4677241.09,5581971.49
3,APAC,Enterprise,274177.24,259736.40
4,Europe,Enterprise,16187115.29,10730427.42


In [220]:
# Output 2. keep only the segments that experienced a decline in sales in all 3 regions

df['var'] = (df['Sales in Euros2023'] < df['Sales in Euros2022']).astype(int)
df = df.groupby('Segment')['var'].sum().reset_index()
df = df[df['var'] == 3][['Segment']]

df

,Segment
1,Enterprise
3,Midmarket
